# Clean HubDailyUsers

Cleans a raw `HubDailyUsers_yyyy-mm-dd.csv` export.

**Deviations / interpretations, mirroring the approach in `Clean_HubMonthlyUsers.ipynb` and `Clean_HubDailyContent.ipynb`:**
- *"Remove completely blank rows"* is evaluated against the `LS_COLS` fields only, not the full raw column set — same reasoning as the other two notebooks: raw pipeline-metadata columns (`DataSource`, `PipelineRunID`, `FileName`, ...) are dropped later and would otherwise mask genuinely blank rows. In this dataset it drops 0 rows on its own.
- The notes say to remove rows where `Date` and/or `CompanyCode` are blank, but per explicit instruction this notebook only drops rows where `Date` is blank — `CompanyCode` being blank is not, on its own, a reason to drop a row.
- The notes' step 6 ("Locate the input file `HubDailyContentData_yyyy-mm-dd.csv`") names the wrong dataset — it's a copy/paste leftover from the Daily Content notes. This notebook locates `HubDailyUsers_yyyy-mm-dd.csv`, matching both the actual input file and the notes' own "Save the dataset" section.
- `SessionDuration` (raw) is dropped, and `SessionDurationInSeconds` (raw) is renamed to `SessionDuration`, per the notes. Despite its name, the raw `SessionDurationInSeconds` column actually holds `hh:mm:ss`-formatted strings, not a count of seconds — the notes call this out (step 30) as an observation; this notebook leaves it as a string and does not attempt to convert or validate it further.
- The notes say to read with `engine="python", escapechar="\\"`, but this notebook does **not** — that combination silently corrupted a real company name (`TBWA\RAAD` → `TBWARAAD` in 5 rows) because this file has no HTML content actually needing backslash-escaped quotes, so `escapechar` only ever strips a literal backslash it shouldn't. A plain read is correct here (see the note on the read cell below).
- The file is read and written with `encoding="utf-8"` explicitly, matching the other two notebooks' reasoning (avoiding `cp1252` corruption of accented characters).


## Imports

In [1]:
import csv
import re
from pathlib import Path

import pandas as pd


## Schema constants

- `LS_COLS` — the final column set and order for the cleaned output.
- `LS_STRING_COLS` — the free-text columns that get whitespace-trimmed.
- `LS_INT_COLS` — the numeric columns that get cast to integer type.
- `SORT_COLS` — the columns (and order) used for the final ascending sort.
- `FILENAME_RE` — extracts the year/month from the input filename, used both to build `month_tag` and to auto-detect the input file.
- `GROUP_COLS`/`SUM_COLS` — used to collapse duplicate rows (see "Collapse duplicate rows" below): every `LS_COLS` field except `SUM_COLS` is a group-by key; `SUM_COLS` (`Users`, `Sessions`, `SessionDuration`) is what gets summed.


In [2]:
LS_COLS = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "DeviceCategory", "UserType", "Users", "Sessions", "SessionDuration",
]
LS_STRING_COLS = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "DeviceCategory", "UserType",
]
LS_INT_COLS = ["Sessions", "Users"]
SORT_COLS = ["Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode"]

FILENAME_RE = re.compile(r"^HubDailyUsers_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS field except the summed
# measures and summing those. SessionDuration is a hh:mm:ss string, not numeric -- see
# the hms helpers in "Collapse duplicate rows" below.
SUM_COLS = ["Users", "Sessions", "SessionDuration"]
GROUP_COLS = [c for c in LS_COLS if c not in SUM_COLS]


## Locate the input file

`find_default_input` looks for a single `HubDailyUsers_yyyy-mm-dd.csv` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.


In [3]:
def find_default_input(directory: Path) -> Path:
    matches = sorted(p for p in directory.glob("HubDailyUsers_*.csv") if FILENAME_RE.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No HubDailyUsers_yyyy-mm-dd.csv file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


## Derive `month_tag` from the filename

The output name has the format of `HubDailyUsers_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one), per the notes' worked example.


In [4]:
def month_tag_from_filename(path: Path) -> str:
    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern HubDailyUsers_yyyy-mm-dd.csv")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


## Cleaning logic

The core transformation, in the order implemented (the notes list these unordered):

1. Drop the raw `SessionDuration` column and rename `SessionDurationInSeconds` → `SessionDuration` (see the note at the top of this notebook).
2. Drop rows that are blank across every `LS_COLS` field.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS`.
6. Trim surrounding whitespace on the `LS_STRING_COLS` fields.
7. Cast `Sessions`, `Users` to integer type.
8. Sort ascending by `Date`, `CompanyCode`, `CompanyName`, `Country`, `HomeCountry`, `HomeCountryCode`.


In [5]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=["SessionDuration"]).rename(columns={"SessionDurationInSeconds": "SessionDuration"})

    # Same reasoning as the other two notebooks: judge "completely blank" against the
    # LS_COLS fields only, since raw pipeline-metadata columns (DataSource, PipelineRunID,
    # FileName, ...) are dropped later and would otherwise mask genuinely blank rows.
    present_ls_cols = [c for c in LS_COLS if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    # Only Date being blank drops a row; a blank CompanyCode alone is kept.
    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[[c for c in LS_COLS if c in df.columns]]

    for col in LS_STRING_COLS:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS, ascending=True).reset_index(drop=True)

    return df


## Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS` value are collapsed into one row, summing `Users` and `Sessions` as integers. `SessionDuration` is a `hh:mm:ss` string (see the note at the top of this notebook), so it's summed by converting each value to total seconds, adding those, then formatting back to `hh:mm:ss` — hours are left unbounded (not wrapped at 24h) since this is an accumulated duration, not a clock time.


In [6]:
def _hms_to_seconds(value: str) -> int:
    h, m, s = value.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


def _seconds_to_hms(total_seconds: int) -> str:
    h, remainder = divmod(total_seconds, 3600)
    m, s = divmod(remainder, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def collapse_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Users"] = df["Users"].astype(int)
    df["Sessions"] = df["Sessions"].astype(int)
    df = df.groupby(GROUP_COLS, as_index=False).agg({
        "Users": "sum",
        "Sessions": "sum",
        "SessionDuration": lambda s: _seconds_to_hms(sum(_hms_to_seconds(v) for v in s)),
    })
    return df[LS_COLS]


## Configure the input file

Leave `INPUT_FILE` as `None` to auto-detect the single raw file in this notebook's `input/` folder, or set it to an explicit path to override (equivalent to the script's optional CLI argument).


In [7]:
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = None  # e.g. "input/HubDailyUsers_2026-08-02.csv"

input_path = Path(INPUT_FILE).resolve() if INPUT_FILE else find_default_input(NOTEBOOK_DIR / "input")
month_tag = month_tag_from_filename(input_path)
input_path, month_tag


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyUsers_2026-08-02.csv'),
 '202607')

## Read the raw CSV

Read everything as strings (`dtype=str`, `keep_default_na=False`) so blank fields and numeric-looking codes pass through unchanged instead of being coerced or turned into `NaN`. `encoding="utf-8"` matches the source file and avoids corrupting accented text.

No `engine="python"`/`escapechar` here — this file has no HTML content needing backslash-escaped quotes, and those options actively corrupted data: the raw file contains a real company name, `TBWA\RAAD` (a genuine single backslash, not a CSV escape artifact), which `escapechar="\\"` silently stripped to `TBWARAAD` in 5 rows. A plain read parses everything else identically and preserves that name correctly.


In [8]:
df_raw = pd.read_csv(input_path, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw.shape


(41702, 21)

## Apply the cleaning steps

In [9]:
df_cleaned = clean(df_raw)
df_cleaned.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,DeviceCategory,UserType,Users,Sessions,SessionDuration
0,2026-07-01 00:00:00.000,,,Algeria,,,,Mobile,Returning User,1,1,00:00:00
1,2026-07-01 00:00:00.000,,,Algeria,,,,Desktop,Returning User,1,1,00:00:00
2,2026-07-01 00:00:00.000,,,Algeria,,null,,Mobile,Returning User,1,1,00:00:00
3,2026-07-01 00:00:00.000,,,Angola,,,,Desktop,Returning User,1,1,00:13:38
4,2026-07-01 00:00:00.000,,,Argentina,,,,Desktop,Returning User,1,1,00:00:00


## Collapse duplicate rows before saving

`rows_before_dedup` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [10]:
rows_before_dedup = len(df_cleaned)
df_cleaned = collapse_duplicates(df_cleaned)
duplicates_collapsed = rows_before_dedup - len(df_cleaned)

print(f"Collapsed {duplicates_collapsed} duplicate rows -> {len(df_cleaned)} rows remaining")


Collapsed 12657 duplicate rows -> 29043 rows remaining


## Save the cleaned dataset

Written as `;`-delimited UTF-8 with minimal quoting, matching the input file's own semicolon delimiter as required by the notes. Saved to this notebook's `output/` folder.


In [11]:
output_dir = NOTEBOOK_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"HubDailyUsers_{month_tag}_cleaned.csv"
df_cleaned.to_csv(output_path, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned)} rows -> {output_path}")


Cleaned 29043 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyUsers_202607_cleaned.csv


## Write summary report

Writes a plain-text report answering: which input file was read, the raw and cleaned row counts, how many duplicate rows exist in each of the raw and cleaned dataframes, the derived `month_tag`, how many rows were dropped for being blank / missing their key fields, and the output CSV's name, followed (after three blank lines) by `df_cleaned.describe()`. Saved to this notebook's `reports/` folder as `HubDailyUsers_<month_tag>_report.txt`.

Duplicate counts use pandas' default `duplicated()` (`keep="first"`), i.e. the number of rows that are repeats of an earlier row — how many rows would go away if the dataframe were deduplicated.


In [12]:
report_lines = [
    f"Input file: {input_path.name}",
    f"Raw row count: {len(df_raw)}",
    f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
    "=======================================================================",
    f"Month tag: {month_tag}",
    f"Blank/missing-key rows dropped: {len(df_raw) - rows_before_dedup}",
    f"Duplicate rows collapsed: {duplicates_collapsed}",
    "=======================================================================",
    f"Cleaned row count: {len(df_cleaned)}",
    f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
    f"Output file: {output_path.name}",
]
report_text = "\n".join(report_lines) + "\n"
report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

reports_dir = NOTEBOOK_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / f"HubDailyUsers_{month_tag}_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print(report_text)
print(f"Report written -> {report_path}")


Input file: HubDailyUsers_2026-08-02.csv
Raw row count: 41702
Raw duplicate rows: 1641
Month tag: 202607
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 12657
Cleaned row count: 29043
Cleaned duplicate rows: 0
Output file: HubDailyUsers_202607_cleaned.csv



              Users      Sessions
count  29043.000000  29043.000000
mean       2.579279      5.165513
std       10.599625     23.167354
min        1.000000      1.000000
25%        1.000000      1.000000
50%        1.000000      1.000000
75%        2.000000      3.000000
max      463.000000   1053.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\HubDailyUsers_202607_report.txt
